In [25]:
import pandas as pd

path = "/Users/simal/Downloads/adult.csv"
df = pd.read_csv(path)

df.head()

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,?,186061,Some-college,10,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K


In [26]:
import numpy as np

df = df.apply(lambda x: x.str.strip() if x.dtype == "object" else x)
df.replace('?', np.nan, inplace=True)

df.dropna(inplace=True)

df.head()

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K
5,34,Private,216864,HS-grad,9,Divorced,Other-service,Unmarried,White,Female,0,3770,45,United-States,<=50K
6,38,Private,150601,10th,6,Separated,Adm-clerical,Unmarried,White,Male,0,3770,40,United-States,<=50K


In [27]:
# make our target income >50K 1 else 0
df['income'] = df['income'].apply(lambda x: 1 if '>50K' in str(x) else 0)

# make male 1 female 0
df['sex'] = df['sex'].apply(lambda x: 1 if 'Male' in str(x) else 0)

df.head()

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,0,0,4356,18,United-States,0
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,0,0,3900,40,United-States,0
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,0,0,3900,40,United-States,0
5,34,Private,216864,HS-grad,9,Divorced,Other-service,Unmarried,White,0,0,3770,45,United-States,0
6,38,Private,150601,10th,6,Separated,Adm-clerical,Unmarried,White,1,0,3770,40,United-States,0


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# target column
y = df['income']
X_raw = df.drop('income', axis=1)

X_encoded = pd.get_dummies(X_raw)
X_encoded = X_encoded.astype(int)


X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

# make the model and train it
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# check accuracy
print(f"Accuracy: {model.score(X_test, y_test):.2f}")

Accuracy: 0.84


In [34]:
from fairlearn.metrics import MetricFrame, selection_rate, demographic_parity_difference
from sklearn.metrics import accuracy_score, recall_score

# get predictions from the model
y_pred = model.predict(X_test)

# deifine the metrics
# Selection Rate: The percentage of people who the model predicted that will get >50K
# Recall: The model's success in actually finding people who got >50K
metrics = {
    'accuracy': accuracy_score,
    'selection_rate': selection_rate,
    'recall': recall_score
}

# analyze, using the 'sex' as sensitive feature
mf = MetricFrame(metrics=metrics,
                 y_true=y_test,
                 y_pred=y_pred,
                 sensitive_features=X_test['sex'])

print("\n--- RESULTS (FEMALE: 0 MALE:1) ---")
print(mf.by_group)

print(f"\nDemographic Parity Difference: {demographic_parity_difference(y_test, y_pred, sensitive_features=X_test['sex']):.4f}")


--- RESULTS (FEMALE: 0 MALE:1) ---
     accuracy  selection_rate    recall
sex                                    
0    0.928571        0.085821  0.561086
1    0.805629        0.269666  0.622361

Demographic Parity Difference: 0.1838


In [ ]:
#overall Accuracy
overall_acc = accuracy_score(y_test, y_pred)

#overall Selection Rate (Percentage of 1s predicted)
overall_selection = y_pred.mean() 

#overall Recall
overall_recall = recall_score(y_test, y_pred)

print(f"Overall Accuracy: {overall_acc:.4f}")
print(f"Overall Selection Rate: {overall_selection:.4f}")
print(f"Overall Recall: {overall_recall:.4f}")

Overall Accuracy: 0.8439
Overall Selection Rate: 0.2125
Overall Recall: 0.6133


# Phase 1: Baseline Model Performance & Fairness Audit

## 1. Experimental Setup
I trained a **Random Forest Classifier** on the UCI Census Income dataset (Adult Dataset) using all available features after proper cleaning and one-hot encoding. My objective was to predict whether an individual earns more than $50K/year.

## 2. Global Performance
- **Test Accuracy:** 84%

## 3. Fairness Metrics (Sensitive Feature: Sex)
Using the `Fairlearn` library, I analyzed the model's behavior across male (1) and female (0) groups:

| Metric | Female (0) | Male (1) | Overall |
| :--- | :--- | :--- | :--- |
| **Accuracy** | 92.86% | 80.56% | 84.39% |
| **Selection Rate** | 8.58% | 26.97% | 21.25% |
| **Recall** | 56.11% | 62.24% | 61.33% |



## 4. Key Observations & Ethical Audit
- There is a significant disparity of **18.38%** in the selection rates. The model is nearly **3 times more likely** to predict a high income for a male applicant than a female applicant.
- The higher accuracy for females (92.86%) is misleading; it stems from the fact that most women in the dataset are in the lower-income bracket. The model defaults to "Low Income" for women, which is safer for accuracy but unfair for high-achieving individuals.
- The lower **Recall** for women (56.11% vs 62.24%) indicates that the model is more prone to **False Negatives** for female candidates, failing to recognize qualified women more often than men.

# Phase 2: Bias Mitigation using Manual Reweighing

## 1. Concept: Correcting Historical Injustice
The baseline model suffers from **algorithmic bias** because it was trained on historical data where women were systematically underrepresented in high-income brackets. **Reweighing** is a pre-processing technique that assigns a mathematical "weight" to each training sample to balance the scales before the model learns.

## 2. The Logic: Independence vs. Correlation
In a perfectly "fair" world, **Gender ($G$)** and **Income ($T$)** should be independent variables. 
- **The Fair World ($P_{expected}$):** The probability of being a high-earner shouldn't depend on gender.
  $$P_{expected} = P(Group) \times P(Target)$$
- **The Biased Reality ($P_{observed}$):** The actual probability we see in the 1994 Census data.
  $$P_{observed} = P(Group \cap Target)$$

## 3. The Mathematical Weight ($W$)
To correct the bias, I calculate a weight for every individual in the training set based on their subgroup (e.g., High-Earning Female):
$$W = \frac{P(Group) \times P(Target)}{P(Group \cap Target)}$$



### **How the weights affect the Model:**
- **High-Earning Women:** Since $P_{observed} < P_{expected}$, their weight will be **$> 1$**. The model treats one such case as if it were multiple people, making their features more "expensive" to ignore.
- **High-Earning Men:** Since $P_{observed} > P_{expected}$, their weight will be **$< 1$**. Their influence is slightly reduced to prevent the model from over-relying on "Male" as a proxy for "Success."



---
*Next Step: I will apply these weights to the `RandomForestClassifier` using the `sample_weight` parameter.*

In [ ]:
def calculate_weights(X, y, sensitive_col):
    """
    Calculates weights for each subgroup to achieve statistical parity.
    Formula: W = [P(Group) * P(Target)] / P(Group & Target)
    """
    df_temp = X.copy()
    df_temp['target'] = y
    df_temp['group'] = X[sensitive_col]
    
    n_total = len(df_temp)
    weights = np.zeros(n_total)
    
    #identify unique groups (0, 1) and targets (0, 1)
    groups = df_temp['group'].unique()
    targets = df_temp['target'].unique()
    
    for g in groups:
        for t in targets:
            # P(Group)
            p_group = len(df_temp[df_temp['group'] == g]) / n_total
            # P(Target)
            p_target = len(df_temp[df_temp['target'] == t]) / n_total
            # P(Group & Target) - The actual observed probability
            p_joint = len(df_temp[(df_temp['group'] == g) & (df_temp['target'] == t)]) / n_total
            
            #calculate the weight for this specific subgroup
            #this balances the 'importance' of the group in the eyes of the model
            subgroup_weight = (p_group * p_target) / p_joint
            
            #apply weight to all rows belonging to this subgroup
            mask = (df_temp['group'] == g) & (df_temp['target'] == t)
            weights[mask] = subgroup_weight
            
            print(f"Group {g}, Target {t}: Weight = {subgroup_weight:.4f}")
            
    return weights

# 1. Generate the weights based on the 'sex' column in the training set
train_weights = calculate_weights(X_train, y_train, 'sex')

# 2. Train a new Random Forest using these weights
# sample_weight tells the algorithm to prioritize the 'underweighted' samples
model_fair = RandomForestClassifier(n_estimators=100, random_state=42)
model_fair.fit(X_train, y_train, sample_weight=train_weights)

# 3. Predict on the test set
y_pred_fair = model_fair.predict(X_test)

# 4. Evaluate the new 'Fair' model
metrics = {
    'accuracy': accuracy_score,
    'selection_rate': selection_rate,
    'recall': recall_score
}

mf_fair = MetricFrame(metrics=metrics,
                      y_true=y_test,
                      y_pred=y_pred_fair,
                      sensitive_features=X_test['sex'])

print("\n--- FAIR MODEL RESULTS (0: Female, 1: Male) ---")
print(mf_fair.by_group)

# 5. Check the Final Demographic Parity Difference
final_dpd = demographic_parity_difference(y_test, y_pred_fair, sensitive_features=X_test['sex'])
print(f"\nNew Demographic Parity Difference: {final_dpd:.4f}")

Group 1, Target 0: Weight = 1.0970
Group 1, Target 1: Weight = 0.7894
Group 0, Target 0: Weight = 0.8464
Group 0, Target 1: Weight = 2.2094

--- FAIR MODEL RESULTS (0: Female, 1: Male) ---
     accuracy  selection_rate    recall
sex                                    
0    0.929638        0.084755  0.561086
1    0.807554        0.272071  0.629398

New Demographic Parity Difference: 0.1873


In [43]:
# Predict on the Training set (where the weights were applied)
y_train_pred_fair = model_fair.predict(X_train)

# Calculate Demographic Parity for the TRAINING set
train_dpd = demographic_parity_difference(y_train, y_train_pred_fair, sensitive_features=X_train['sex'])

print(f"Training Demographic Parity Difference: {train_dpd:.4f}")
print(f"Test Demographic Parity Difference: {final_dpd:.4f}") # This was 0.1873

Training Demographic Parity Difference: 0.2026
Test Demographic Parity Difference: 0.1873


In [ ]:
import matplotlib.pyplot as plt

#get feature importances from the fair model
importances = model_fair.feature_importances_
feature_names = X_train.columns
feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False).head(10)

print(feature_importance_df)

                              Feature  Importance
1                              fnlwgt    0.160138
0                                 age    0.143996
4                        capital.gain    0.103894
6                      hours.per.week    0.080305
32  marital.status_Married-civ-spouse    0.067580
2                       education.num    0.066546
56                  relationship_Wife    0.038528
5                        capital.loss    0.031101
40         occupation_Exec-managerial    0.022051
51               relationship_Husband    0.019069


## **Phase 2 Audit: The "Resistance" of the Random Forest Model**

### **1. Reweighing Implementation & Subgroup Weights**
To mitigate the bias observed in Phase 1, I calculated manual weights to achieve **Statistical Parity**. The goal was to "boost" the importance of underrepresented successful groups during training:
* **High-Earning Females (Group 0, Target 1):** Weight = **2.2094** (Significant boost)
* **High-Earning Males (Group 1, Target 1):** Weight = **0.7894** (Reduced influence)

### **2. Results: The Fairness-Accuracy Paradox**
Despite a massive 2.2x boost to high-earning women, the fairness metrics showed no improvement—and in some cases, the gap widened:

| Metric | Baseline Parity | Mitigated Parity (Test) | Mitigated Parity (Train) |
| :--- | :--- | :--- | :--- |
| **Demographic Parity Diff** | 0.1838 | **0.1873** | **0.2026** |

**Observation:** The **Demographic Parity Difference** is higher on the Training set (20.26%) than the Test set (18.73%). This might indicate that the model used its high complexity to "overfit" to the specific weighted samples rather than learning a generalized fair rule.

### **3. Root Cause Analysis: Proxy Variables & Feature Importance**
The failure of Reweighing can be explained by the **Feature Importance** ranking of the mitigated model. The Random Forest bypassed the "Gender" weights by utilizing highly correlated **Proxy Variables**:

1. **Direct Proxies:** `relationship_Wife` and `relationship_Husband` appear in the top 10 features. These columns are mathematically synonymous with gender in this dataset. 
2. **Socio-Economic Proxies:** `marital.status_Married-civ-spouse` ranks 5th in importance. In the 1994 Census context, marriage status is a strong proxy for gendered income disparities.
3. **The "Greedy" Nature of Trees:** Random Forests are designed to minimize impurity. If a proxy like "Husband" provides a cleaner split than the weighted "Sex" column, the algorithm will prioritize the proxy, effectively "tunneling" around our fairness constraints.



### **4. Conclusion for Phase 2**
Reweighing (a Pre-processing technique) proved insufficient for a non-linear ensemble like Random Forest due to the presence of strong redundant encodings (proxies). 

In [ ]:
# 1. Identify all columns related to Relationship and Marital Status
#these are the strongest proxies for Gender in this dataset
proxy_columns = [col for col in X_encoded.columns if 'relationship' in col or 'marital.status' in col]

print(f"Dropping {len(proxy_columns)} proxy columns: {proxy_columns}")

# 2. Create the new 'Filtered' feature set
X_filtered = X_encoded.drop(columns=proxy_columns)

# 3. Split the filtered data
X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(X_filtered, y, test_size=0.2, random_state=42)

# 4. Train the Random Forest on the filtered data (without weights first)
model_filtered = RandomForestClassifier(n_estimators=100, random_state=42)
model_filtered.fit(X_train_f, y_train_f)

# 5. Check the new Baseline (Filtered)
y_pred_f = model_filtered.predict(X_test_f)
print(f"Filtered Model Accuracy: {model_filtered.score(X_test_f, y_test_f):.4f}")

Dropping 13 proxy columns: ['marital.status_Divorced', 'marital.status_Married-AF-spouse', 'marital.status_Married-civ-spouse', 'marital.status_Married-spouse-absent', 'marital.status_Never-married', 'marital.status_Separated', 'marital.status_Widowed', 'relationship_Husband', 'relationship_Not-in-family', 'relationship_Other-relative', 'relationship_Own-child', 'relationship_Unmarried', 'relationship_Wife']
Filtered Model Accuracy: 0.8258


In [46]:

metrics = {'accuracy': accuracy_score, 'selection_rate': selection_rate}

mf_filtered = MetricFrame(metrics=metrics,
                          y_true=y_test_f,
                          y_pred=y_pred_f,
                          sensitive_features=X_test_f['sex'])

print("\n--- FILTERED MODEL RESULTS (No Proxies) ---")
print(mf_filtered.by_group)
print(f"Filtered Parity Difference: {demographic_parity_difference(y_test_f, y_pred_f, sensitive_features=X_test_f['sex']):.4f}")


--- FILTERED MODEL RESULTS (No Proxies) ---
     accuracy  selection_rate
sex                          
0    0.910981        0.059701
1    0.787347        0.255713
Filtered Parity Difference: 0.1960


## **Phase 3: Fairness Through Blindness (Feature Dropping)**

### **1. Methodology**
Based on the Feature Importance ranking from Phase 2, I identified and removed all "Proxy" features related to `relationship` and `marital-status`. The hypothesis was that removing these "backdoors" would force the model to rely on merit-based features (education, capital gain).

### **2. The Paradoxical Result**
Surprisingly, removing these features **increased** the Demographic Parity Difference:

| Metric | Baseline (All Features) | Filtered (No Proxies) |
| :--- | :--- | :--- |
| **Accuracy (Overall)** | 84.39% | **~82%** |
| **Selection Rate (Female)** | 8.58% | **5.97%** |
| **Selection Rate (Male)** | 26.97% | **25.57%** |
| **Parity Difference** | 0.1838 | **0.1960** |

### **3. Technical Diagnostic: Why Blindness Failed**
1. Gender bias is "smeared" across almost every feature in the 1994 Census data. Removing the most obvious ones just forced the Random Forest to find more subtle, non-linear combinations of the remaining features to maintain accuracy.
2. By removing `relationship_Wife`, I may have removed a feature that actually helped the model identify high-earning women. Without it, the model defaults to a "Low Income" prediction for women to maximize overall accuracy.
3. **Conclusion:** Simply "hiding" sensitive information from a complex model (Random Forest) is ineffective and can be counter-productive.

In [ ]:
# 1. Calculate weights specifically for the FILTERED training set
train_weights_f = calculate_weights(X_train_f, y_train_f, 'sex')

# 2. Train the Random Forest on Filtered Data WITH Weights
model_fair_filtered = RandomForestClassifier(n_estimators=100, random_state=42)
model_fair_filtered.fit(X_train_f, y_train_f, sample_weight=train_weights_f)

# 3. Predict on the Filtered Test Set
y_pred_ff = model_fair_filtered.predict(X_test_f)

# 4. Final Fairness Audit for this hybrid approach
mf_ff = MetricFrame(metrics=metrics,
                    y_true=y_test_f,
                    y_pred=y_pred_ff,
                    sensitive_features=X_test_f['sex'])

print("\n--- HYBRID MODEL RESULTS (No Proxies + Reweighing) ---")
print(mf_ff.by_group)
print(f"Hybrid Parity Difference: {demographic_parity_difference(y_test_f, y_pred_ff, sensitive_features=X_test_f['sex']):.4f}")

Group 1, Target 0: Weight = 1.0970
Group 1, Target 1: Weight = 0.7894
Group 0, Target 0: Weight = 0.8464
Group 0, Target 1: Weight = 2.2094

--- HYBRID MODEL RESULTS (No Proxies + Reweighing) ---
     accuracy  selection_rate
sex                          
0    0.904584        0.067164
1    0.794082        0.247053
Hybrid Parity Difference: 0.1799


## **Phase 4: Hybrid Mitigation (No Proxies + Reweighing)**

### **1. Strategy**
I combined **Pre-processing (Reweighing)** with **Feature Selection (Proxy Dropping)**. By removing the marital/relationship "backdoors," I aimed to force the model to utilize the increased weights of the unprivileged group through merit-based features.

### **2. Results**
For the first time, I observed a reduction in the parity gap compared to the original baseline:

| Metric | Baseline | Filtered (No Proxies) | Hybrid (Filtered + Weights) |
| :--- | :--- | :--- | :--- |
| **Parity Difference** | 0.1838 | 0.1960 | **0.1799** |
| **Female Selection Rate**| 8.58% | 5.97% | **6.71%** |

### **3. Analysis**
- **Synergy:** Reweighing is more effective when proxy variables are removed, as the model cannot easily circumvent the weights.
- **Limitation:** The marginal improvement (from 0.18 to 0.17) suggests that the Random Forest algorithm is inherently biased toward the dominant patterns in the data. Its non-linear nature allows it to reconstruct "gender" from other features (e.g., occupation, hours worked).


In [ ]:
# 1. Get the raw probabilities from the Hybrid Model
#column 1 is the probability of earning >50K
probs_hybrid = model_fair_filtered.predict_proba(X_test_f)[:, 1]

# 2. Create a results dataframe to play with thresholds
results_hybrid = pd.DataFrame({
    'sex': X_test_f['sex'].values,
    'prob': probs_hybrid,
    'true_label': y_test_f.values
})

# 3. Apply CUSTOM THRESHOLDS
# I will "tune" these to close the 17.99% gap
fem_threshold = 0.32  # Lowering the bar for women
male_threshold = 0.62 # Raising the bar for men

y_post_preds = []
for idx, row in results_hybrid.iterrows():
    if row['sex'] == 0: # Female
        y_post_preds.append(1 if row['prob'] >= fem_threshold else 0)
    else: # Male
        y_post_preds.append(1 if row['prob'] >= male_threshold else 0)

# 4. Final Fairness Audit

new_dpd_post = demographic_parity_difference(y_test_f, y_post_preds, sensitive_features=X_test_f['sex'])
print(f"Post-Processed Parity Difference: {new_dpd_post:.4f}")

# Check selection rates
sr_fem = np.mean([p for i, p in enumerate(y_post_preds) if X_test_f.iloc[i]['sex'] == 0])
sr_male = np.mean([p for i, p in enumerate(y_post_preds) if X_test_f.iloc[i]['sex'] == 1])
print(f"Female Selection Rate: {sr_fem:.4f}")
print(f"Male Selection Rate: {sr_male:.4f}")

Post-Processed Parity Difference: 0.0507
Female Selection Rate: 0.1317
Male Selection Rate: 0.1823


## **Phase 5: Post-Processing (Threshold Optimization)**

### **1. Strategy**
Having observed that Pre-processing (Reweighing) and Feature Selection (Proxy Dropping) were insufficient to overcome the structural bias in the Random Forest, I implemented **Post-Processing**. I abandoned the global 0.5 decision threshold in favor of **Group-Specific Thresholds**:
* **Female Threshold:** 0.32 (Lowered to compensate for suppressed scores)
* **Male Threshold:** 0.62 (Raised to mitigate historical over-representation)

### **2. Final Comparison**

| Stage | Parity Difference | Female Selection Rate | Male Selection Rate |
| :--- | :--- | :--- | :--- |
| **Baseline** | 18.38% | 8.58% | 26.96% |
| **Hybrid (Filtered + Weights)** | 17.99% | 6.71% | 24.70% |
| **Post-Processed (Final)** | **5.07%** | **13.17%** | **18.23%** |

### **3. Conclusion & Ethical Trade-offs**
- **Success:** I successfully reduced the Demographic Parity Difference by **~72%** compared to the baseline.
- **Selection Balance:** The selection rate for women nearly doubled, moving from 6.7% to 13.17%, creating a much more equitable distribution of outcomes.


In [ ]:
from sklearn.metrics import accuracy_score, recall_score, classification_report

# 1. Calculate Accuracy of the Post-Processed predictions
post_accuracy = accuracy_score(y_test_f, y_post_preds)

# 2. Calculate Recall (how many high-earners did the model actually find?)
post_recall = recall_score(y_test_f, y_post_preds)

print(f"--- FINAL MODEL PERFORMANCE ---")
print(f"Original Baseline Accuracy: 0.8439")
print(f"Post-Processed Accuracy:   {post_accuracy:.4f}")
print(f"\nOriginal Baseline Recall:   0.6133")
print(f"Post-Processed Recall:     {post_recall:.4f}")

# 3. Detailed breakdown
print("\nClassification Report:")
print(classification_report(y_test_f, y_post_preds))

--- FINAL MODEL PERFORMANCE ---
Original Baseline Accuracy: 0.8439
Post-Processed Accuracy:   0.8158

Original Baseline Recall:   0.6133
Post-Processed Recall:     0.4647

Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.93      0.88      4533
           1       0.69      0.46      0.56      1500

    accuracy                           0.82      6033
   macro avg       0.77      0.70      0.72      6033
weighted avg       0.80      0.82      0.80      6033



## **Final Audit: The Accuracy-Fairness Trade-off**

After implementing **Post-Processing Thresholds**, I achieved a "Fairer" model, but the classification metrics reveal the inherent tension in bias mitigation:

### **1. Performance Comparison**
| Metric | Original Baseline | Post-Processed (Final) | Change |
| :--- | :--- | :--- | :--- |
| **Accuracy** | 0.8439 | 0.8158 | -2.81% |
| **Recall (Overall)** | 0.6133 | 0.4647 | -14.86% |
| **Parity Difference** | 0.1838 | **0.0507** | **-13.31%** |

### **2. Technical Interpretation**
-  To achieve a **5% Parity Difference**, I accepted a ~3% drop in accuracy. This is the "Fairness Tax" required to decouple the model from historical socio-economic biases.
-  The drop in recall indicates that by raising the threshold for the privileged group (males), we are intentionally missing some high-earners to ensure the **Selection Rate** is balanced across genders.
